# Phase 2C — VideoMAE fine-tuning on UCF-Crime (Colab GPU)

Fine-tunes `MCG-NJU/videomae-base-finetuned-kinetics` with a fresh 14-class head on the **official** UCF-Crime Action Recognition split (Fold 2: `train_002.txt` / `test_002.txt`). No random splits, ever.

**Safe to run first (cells 1–10):** environment check, install, Drive mount, path config, split validation, data smoke test, model load, head check, 20/10-video 1-epoch sanity training + eval. **Cell 11 (full Fold-2 training) is gated** — it refuses to run until you explicitly set `RUN_FULL_TRAINING = True`.

In [ ]:
# Cell 1 - Environment check (need a T4 GPU: Runtime > Change runtime type)
import sys
import torch
print('python:', sys.version.split()[0])
print('torch:', torch.__version__, '| cuda_available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))
    free, total = torch.cuda.mem_get_info()
    print(f'gpu memory: {free / 1e9:.1f} free / {total / 1e9:.1f} total GB')
else:
    raise SystemExit('NO GPU - attach a T4 GPU runtime first')

## 2. Code + dependencies

In [ ]:
# Cell 2 - Clone the project and install Phase 2C dependencies
# (Colab already ships CUDA-enabled torch; do not let pip downgrade it.)
!git clone https://github.com/iesxz-c/Final.git /content/Final
%cd /content/Final
!pip install -q ultralytics transformers pyyaml safetensors huggingface_hub

## 3. Mount Google Drive (videos stay on Drive, never in git)

In [ ]:
# Cell 3 - Mount Drive
from google.colab import drive
drive.mount('/content/drive')

## 4. Configure paths (EDIT to match your Drive layout)

In [ ]:
# Cell 4 - Point the project at Drive. Outputs also live on Drive so a
# Colab interruption never destroys checkpoints.
import os
import pathlib
DATASET_ROOT = '/content/drive/MyDrive'  # <-- EDIT if your folders live elsewhere
ANOMALY_ROOT = f'{DATASET_ROOT}/Anomaly-Videos-Part-1/Anomaly-Videos-Part-1'
NORMAL_ROOT = f'{DATASET_ROOT}/Normal_Videos_for_Event_Recognition/Normal_Videos_for_Event_Recognition'
SPLIT_ROOT = '/content/Final/UCF_Crimes-Train-Test-Split/Action_Regnition_splits'
OUTPUT_ROOT = f'{DATASET_ROOT}/ucf-crime-output/phase2c'
os.environ['CCTV_ANOMALY_VIDEOS_DIR'] = ANOMALY_ROOT
os.environ['CCTV_NORMAL_VIDEOS_DIR'] = NORMAL_ROOT
assert pathlib.Path(ANOMALY_ROOT).is_dir(), f'missing: {ANOMALY_ROOT}'
assert pathlib.Path(NORMAL_ROOT).is_dir(), f'missing: {NORMAL_ROOT}'
assert pathlib.Path(SPLIT_ROOT).is_dir(), f'missing: {SPLIT_ROOT} (pull latest repo)'
print('anomaly:', ANOMALY_ROOT)
print('normal :', NORMAL_ROOT)
print('splits :', SPLIT_ROOT)
print('output :', OUTPUT_ROOT)

## 5. Validate the dataset and official split 002

In [ ]:
# Cell 5 - Expect: train=532 (effective 531, Arson019 absent), test=168
# (effective 167), train/test overlap 0.
!python -m src.pipeline.ucf_splits --split 002

## 6. Tiny dataset sanity check (decodes 2 real videos, no model)

In [ ]:
# Cell 6 - Builds one train batch and one eval batch; prints tensor shapes.
# Expect e.g. train batch=(1, 3, 16, 224, 224).
!python -m src.pipeline.activity_data --split 002 --num-videos 2 --num-eval-clips 2

## 7. Load VideoMAE (downloads the HF checkpoint once, then caches it)

In [ ]:
# Cell 7 - Pretrained backbone + fresh 14-class head on CUDA. No training.
import torch
from src.pipeline.train_activity import build_model, resolve_device
device = resolve_device('cuda')
model, notes = build_model('MCG-NJU/videomae-base-finetuned-kinetics', device)
print('device:', device)
print('weights:', notes)
del model
torch.cuda.empty_cache()

## 8. Verify the 14-class classification head

In [ ]:
# Cell 8 - Dummy forward must yield logits of shape (1, 14).
import torch
from src.pipeline.train_activity import build_model, resolve_device
from src.pipeline.ucf_splits import CLASS_NAMES
assert len(CLASS_NAMES) == 14, CLASS_NAMES
device = resolve_device('cuda')
model, _ = build_model('MCG-NJU/videomae-base-finetuned-kinetics', device)
model.eval()
with torch.no_grad():
    dummy = torch.zeros(1, 3, 16, 224, 224, device=device)
    logits = model(pixel_values=dummy).logits
assert tuple(logits.shape) == (1, 14), tuple(logits.shape)
print('OK: 14-class head, logits', tuple(logits.shape))
print('labels:', CLASS_NAMES)
del model
torch.cuda.empty_cache()

## 9. Sanity training: 20 train / 10 test videos, 1 epoch

In [ ]:
# Cell 9 - SAFE: tiny end-to-end run proving the training loop works.
!python -m src.pipeline.train_activity --split 002 --device cuda --epochs 1 \
  --batch-size 2 --max-train-videos 20 --max-test-videos 10 \
  --output-dir {OUTPUT_ROOT}/phase2c-sanity

## 10. Evaluate the sanity run

In [ ]:
# Cell 10 - SAFE: video-level metrics for the sanity checkpoint.
!python -m src.pipeline.train_activity --split 002 --device cuda \
  --max-test-videos 10 --eval-only --output-dir {OUTPUT_ROOT}/phase2c-sanity

## 11. FULL Fold-2 training — GATED, does nothing until you opt in

In [ ]:
# Cell 11 - FULL RUN (takes hours on a T4; checkpoints land on Drive).
# To launch: set RUN_FULL_TRAINING = True and execute this cell again.
RUN_FULL_TRAINING = False
assert RUN_FULL_TRAINING, 'Set RUN_FULL_TRAINING = True to start full Fold-2 training.'
!python -m src.pipeline.train_activity --split 002 --device cuda --epochs 10 \
  --batch-size 4 --gradient-accumulation 2 --learning-rate 5e-5 --num-workers 4 \
  --output-dir {OUTPUT_ROOT}/phase2c-fold2

## 12. Evaluate the full Fold-2 run (video-level)

In [ ]:
# Cell 12 - Loads final_model from the full run and scores all 167 test videos.
!python -m src.pipeline.train_activity --split 002 --device cuda \
  --eval-only --output-dir {OUTPUT_ROOT}/phase2c-fold2

## 13. Display metrics

In [ ]:
# Cell 13 - Accuracy, macro P/R/F1, per-class F1, confusion matrix.
import json
m = json.load(open(f'{OUTPUT_ROOT}/phase2c-fold2/metrics_final.json'))
print('accuracy       :', m['accuracy'])
print('macro precision:', m['macro_precision'])
print('macro recall   :', m['macro_recall'])
print('macro F1       :', m['macro_f1'])
print('per-class F1:')
for cls, f1 in m['per_class_f1'].items():
    print(f'  {cls:15s} {f1}')
print('confusion matrix (rows=true):')
for row in m['confusion_matrix']:
    print(' ', row)
try:
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(8, 7))
    ax.imshow(m['confusion_matrix'])
    ax.set_xticks(range(14)); ax.set_yticks(range(14))
    ax.set_xticklabels(list(m['per_class_f1']), rotation=45, ha='right')
    ax.set_yticklabels(list(m['per_class_f1']))
    fig.tight_layout()
    fig.savefig(f'{OUTPUT_ROOT}/phase2c-fold2/confusion_matrix.png')
    print('saved confusion_matrix.png')
except ImportError:
    print('matplotlib unavailable, skipping plot')

## 14. Save final checkpoint + metrics (already on Drive — collect here)

In [ ]:
# Cell 14 - Gather the deliverables into one versioned Drive folder.
import pathlib
import shutil
src = pathlib.Path(f'{OUTPUT_ROOT}/phase2c-fold2')
dst = pathlib.Path(f'{OUTPUT_ROOT}/phase2c-fold2-final')
dst.mkdir(parents=True, exist_ok=True)
for name in ['metrics_final.json', 'label_map.json', 'config_used.json']:
    shutil.copy2(src / name, dst / name)
for name in ['final_model', 'checkpoint-best']:
    if (src / name).exists():
        shutil.copytree(src / name, dst / name, dirs_exist_ok=True)
print('final artifacts:', sorted(p.name for p in dst.iterdir()))